In [28]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 1
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
# -----------------------------------------------------------------------------
def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0):
    n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION — unchanged from before
# -----------------------------------------------------------------------------
F_STOP_HZ = 50e6
CHIRP_SPAN_HZ = 350e6
F_START_HZ = F_STOP_HZ - CHIRP_SPAN_HZ

NUM_STEPS = 125
AMPLITUDE = 0.9

#change both buffer duration and total sweep you can set it to anytime
BUFFER_DURATION_S_REQUESTED = 6e-3
max_feasible_buffer_s = 0.95 * ENV_MAXLEN / ENV_SR
BUFFER_DURATION_S = min(BUFFER_DURATION_S_REQUESTED, max_feasible_buffer_s)

TOTAL_SWEEP_S = 6e-3
STEP_HOLD_S = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

buffer_step_s = BUFFER_DURATION_S / NUM_STEPS
print(f"Each step's stored buffer: {buffer_step_s*1e9:.1f} ns of raw samples, "
      f"repeated via mode='periodic' for {STEP_HOLD_US:.2f} us before switching.")

freqs_hz = np.linspace(F_START_HZ, F_STOP_HZ, NUM_STEPS)

maxv = soccfg.get_maxv(GEN_CH)
idata_list = []
qdata_list = []

phase = 0.0
for f in freqs_hz:
    y, phase, n_samples = serrodyne_tone(f, buffer_step_s, ENV_SR, amplitude=AMPLITUDE, phase0=phase)
    pad = (-len(y)) % samps_per_clk
    if pad:
        y = np.concatenate([y, np.zeros(pad)])
    i_wave = np.round(y * maxv).astype(np.int16)
    q_wave = np.zeros_like(i_wave)
    idata_list.append(i_wave)
    qdata_list.append(q_wave)

samples_per_step = len(idata_list[0])
total_samples = samples_per_step * NUM_STEPS
print(f"Per-step buffer length: {samples_per_step} samples")
print(f"Total envelope samples: {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN, (
    f"Envelope memory exceeded: need {total_samples} samples, "
    f"only {ENV_MAXLEN} available on gen {GEN_CH}."
)
assert samples_per_step % samps_per_clk == 0

# -----------------------------------------------------------------------------
# 4. PROGRAM: a real hardware loop over steps (RAveragerProgram), no unrolling.
#    'expts' = NUM_STEPS is executed entirely in tProc via loopnz — program
#    size no longer scales with NUM_STEPS. Only the 'addr' register changes
#    between steps; freq/phase/gain/mode are written once in initialize().
# -----------------------------------------------------------------------------
class RepeatedStepSerrodyneProgram(RAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i, (idata_step, qdata_step) in enumerate(zip(cfg["idata_list"], cfg["qdata_list"])):
            self.add_envelope(ch=res_ch, name=f"serr_{i}", idata=idata_step, qdata=qdata_step)

        # Seed freq/phase/gain/mode once, using step 0's waveform to get the
        # correct addr + mode word (mode encodes length/outsel, identical for
        # every step since all buffers are the same length).
        self.set_pulse_registers(
            ch=res_ch,
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            waveform="serr_0",
            outsel="input",
            mode="periodic",
        )

        self.r_rp = self.ch_page(res_ch)
        self.r_addr = self.sreg(res_ch, "addr")
        self.addr_step = cfg["samples_per_step"] // self.soccfg["gens"][res_ch]["samps_per_clk"]

        self.synci(200)

    def body(self):
        res_ch = self.cfg["res_ch"]
        step_cycles = self.us2cycles(self.cfg["step_hold_us"])

        self.trigger(pins=[0])            # per-step marker pulse -- scope sync
        self.pulse(ch=res_ch, t='auto')   # reuses freq/phase/gain/mode; picks up CURRENT addr register value
        self.sync_all(step_cycles)        # hold this step's periodic output for the dwell time

    def update(self):
        # This is the actual "switch to next step" -- one instruction, run in
        # hardware between experiment points, not unrolled in Python.
        self.mathi(self.r_rp, self.r_addr, self.r_addr, '+', self.addr_step)

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "reps": 1,
    "expts": NUM_STEPS,
    "idata_list": idata_list,
    "qdata_list": qdata_list,
    "samples_per_step": samples_per_step,
    "step_hold_us": STEP_HOLD_US,
    "gain": 32767,
}

prog = RepeatedStepSerrodyneProgram(soccfg, config)
prog.run(soc)
print(f"Running on hardware — full sequence is {NUM_STEPS} steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS*STEP_HOLD_US*1e-3:.3f} ms total.")
# soc.reset_gens()

Generator 1: f_fabric=614.400 MHz, samps_per_clk=16, envelope sample rate=9.8304 GSPS
Envelope memory available: 65536 samples
Each step's stored buffer: 50.7 ns of raw samples, repeated via mode='periodic' for 48.00 us before switching.
Per-step buffer length: 512 samples
Total envelope samples: 64000 / 65536 available
Running on hardware — full sequence is 125 steps x 48.00 us = 6.000 ms total.


In [ ]:
soc.reset_gens()